# 04 - Difference-in-differences

Difference-in-differences is useful when treatment starts at a specific time for one group and we can compare the change against a control group.


## Causal question
What is the effect of an intervention that starts in the second half of the panel on the unit-level outcome?

## Causal setup
- Treatment: `treatment`, indicating whether a unit is exposed after treatment starts.
- Outcome: `outcome`.
- Covariates: none, by design.
- Unit of analysis: one panel unit (`unit`) observed over repeated `time` periods.

## Estimand
Estimate the ATT: the mean change in outcomes for treated units relative to controls between pre and post periods.

## Identification assumptions
Parallel trends between treated and control groups absent treatment, and stable assignment once treatment starts.

## Uncertainty and limitations
Bootstrap confidence interval around the ATT, plus a placebo pre-period check.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Step execution
This cell performs the next computation. Read the result and tie it back to the causal setup before moving on.


In [ ]:
from causal_inference_lab.data_generators import make_did_panel
from causal_inference_lab.difference_in_differences import difference_in_differences
from causal_inference_lab.uncertainty import bootstrap_ate
from causal_inference_lab.plotting import plot_did_trends

dataset = make_did_panel(n_units=600, n_periods=8, seed=7)
data = dataset.data

data.head()


## Visual trend check

The key assumption is parallel trends: without treatment, the treated and control groups would have followed similar trends.


In [ ]:
fig = plot_did_trends(data)
plt.show()


## Two-way fixed effects-style regression

We estimate the treatment effect using unit and time fixed effects.


In [ ]:
did_result = difference_in_differences(data)
did_bootstrap = bootstrap_ate(
    data=data,
    estimator=lambda frame: difference_in_differences(frame).effect.estimate,
    n_bootstrap_samples=500,
    seed=42,
    confidence_level=0.95,
)

print(f"Estimated DiD ATT:   {did_result.effect.estimate:.3f}")
print(
    f"{int(did_bootstrap.confidence_level*100)}% bootstrap interval: "
    f"[{did_bootstrap.lower:.3f}, {did_bootstrap.upper:.3f}]"
)
print(f"Bootstrap standard error: {did_bootstrap.std_error:.3f}")
print(f"True effect:            {dataset.true_ate:.3f}")
print(f"Pre-trend slope diff:   {did_result.pre_trend_slope_difference:.4f}")
print(f"Pre-trend p-value:      {did_result.pre_trend_p_value:.3f}")


## Placebo pre-period check

In a real project, we would test for differential pre-trends. Here we create a placebo post indicator inside the pre-treatment period.


In [ ]:
pre_data = data[data["time"] < 4].copy()
pre_data["placebo_post"] = (pre_data["time"] >= 2).astype(int)
placebo_result = difference_in_differences(pre_data, post_col="placebo_post")
print(f"Placebo estimate: {placebo_result.effect.estimate:.3f}")
print(f"Placebo p-value:  {placebo_result.pre_trend_p_value:.3f}")


**Interpretation.** A small placebo estimate supports the design, but it does not prove parallel trends. It only provides evidence that the pre-period does not show a large differential shift.
